In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# qwen_rs_caption_prefix_suffix_multisent.py

# Future compatibility imports
from __future__ import annotations
import os, sys, json, re
from pathlib import Path
from typing import List, Tuple
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText



# USER SETTINGS


# Folder containing all aerial images
IMAGES_DIR = "/content/drive/MyDrive/ALL IN_pg/New Work for Journal /images"

# Output JSONL file where captions will be saved
OUT_JSONL = "/content/drive/MyDrive/ALL IN_pg/New Work for Journal /buildings_cap.jsonl"

# overwrite → create new file
# resume → continue from previous output
MODE = "overwrite"

# Maximum number of generated tokens
MAX_NEW_TOKS = 120

# Random seed for reproducibility
SEED = 42


# Optional environment variable overrides
# MODEL_ID=Qwen/Qwen2.5-VL-7B-Instruct
# BATCH_SIZE=12
# QUANTIZE_4BIT=1
# USE_FAST_PROCESSOR=0



# SYSTEM PROMPT

# Defines the role and behavior of the model
SYSTEM_PROMPT = (
    "You are an expert annotator of nadir-view aerial/satellite RGB imagery. "
    "Focus on buildings first: type (detached houses, apartment blocks, industrial sheds), "
    "roof form (flat/gabled/sawtooth), density (sparse/medium/compact), layout/pattern "
    "(grid/irregular/linear), orientation (N/S/E/W), and the urban context (roads, courtyards, "
    "parking, vegetation, water). Be objective and avoid brands or speculation."
)


# USER INSTRUCTION

# Actual task instruction sent to the model
USER_INSTRUCTION = (
    "Overhead RGB orthophoto. Write 3–4 concise sentences strictly about the built environment: "
    "1) State if buildings are present and give a rough count bucket: none / few (1–5) / some (6–20) / many (>20). "
    "2) Describe building types and roof styles plus heights if inferable (low-rise/mid-rise). "
    "3) Summarize spatial pattern & street orientation. "
    "4) Mention one notable feature. "
    "Keep sentences 10-40 words each."
)


# FILE TYPES

# Allowed image extensions
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# Prefix format used for PaLI-Gemma style instruction tuning
PREFIX_STR = "<image>\ncaption en:"


# Sentence constraints
MIN_SENTENCES = 3
MAX_SENTENCES = 4

MIN_WORDS_PER_SENT = 6
MAX_WORDS_PER_SENT = 22


# FIND ALL IMAGES
def find_images(root: str | Path) -> List[Path]:

    # Convert path to Path object
    root = Path(root)

    # Search recursively for valid image files
    return sorted(
        p for p in root.rglob("*")
        if p.suffix.lower() in EXTS
    )


# LOAD IMAGE
def load_image_rgb(p: Path) -> Image.Image | None:

    try:
        # Open image
        img = Image.open(p)

        # Convert image to RGB if needed
        if img.mode != "RGB":
            img = img.convert("RGB")

        return img

    except Exception as e:

        # Skip corrupted/unreadable images
        print(f"[WARN] Could not open {p}: {e}", file=sys.stderr)

        return None


# SPLIT SENTENCES
def split_sentences(t: str) -> List[str]:

    # Split text based on punctuation
    parts = re.split(r'(?<=[.!?])\s+', t.strip())

    return [s.strip() for s in parts if s.strip()]


# BAD MODEL RESPONSES
# Responses we do not want
BAD_STARTS = tuple(s.lower() for s in [
    "describe this image",
    "this image",
    "the image",
    "the photo",
    "as an ai",
    "i'm sorry",
])

BAD_SUBSTRS = tuple(s.lower() for s in [
    "upload",
    "please provide",
    "cannot",
    "sorry",
])


# LIMIT SENTENCE LENGTH
def trim_sentence_words(s: str, max_words: int) -> str:

    words = s.split()

    # If sentence already short enough
    if len(words) <= max_words:
        return s

    # Trim sentence
    trimmed = " ".join(words[:max_words]).rstrip(",;:")

    # Ensure proper punctuation
    if not trimmed.endswith((".", "!", "?")):
        trimmed += "."

    return trimmed


# CLEAN GENERATED TEXT
def rs_clean_multi(text: str) -> str:

    # Remove extra spaces/newlines
    t = text.replace("\r", " ").replace("\n", " ").strip()
    t = " ".join(t.split())

    # Lowercase version for checks
    tl = t.lower()

    # Remove bad/refusal outputs
    if any(b in tl for b in BAD_SUBSTRS):
        return ""

    # Split into sentences
    sents = split_sentences(t)

    if not sents:
        return ""

    cleaned = []

    # Process each sentence
    for s in sents:

        sl = s.lower().lstrip(" ,.;:-")

        # Remove bad sentence starters
        if any(sl.startswith(bs) for bs in BAD_STARTS):
            continue

        ws = sl.split()

        # Skip too-short sentences
        if len(ws) < MIN_WORDS_PER_SENT:
            continue

        # Trim sentence length
        s = trim_sentence_words(s, MAX_WORDS_PER_SENT)

        # Ensure punctuation
        if not s.endswith((".", "!", "?")):
            s += "."

        cleaned.append(s)

    # Require minimum number of sentences
    if len(cleaned) < MIN_SENTENCES:
        return ""

    # Keep only first 4 sentences
    if len(cleaned) > MAX_SENTENCES:
        cleaned = cleaned[:MAX_SENTENCES]

    return " ".join(cleaned)


# PREPARE MODEL CHAT FORMAT
def prepare_batch_messages_rs(images: List[Image.Image], processor):

    texts = []
    imgs = []

    for img in images:

        # Build chat-style prompt
        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },

            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": USER_INSTRUCTION},
                ]
            }
        ]

        # Convert messages into model format
        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        texts.append(text)
        imgs.append(img)

    return texts, imgs


# GENERATE OUTPU
def generate_once(model, inputs, *, max_new_tokens=120,
                  sample=False, beams=3):

    kwargs = dict(max_new_tokens=max_new_tokens)

    # Sampling mode
    if sample:
        kwargs.update(
            dict(
                do_sample=True,
                temperature=0.4,
                top_p=0.9
            )
        )

    # Beam search mode
    else:
        kwargs.update(
            dict(
                do_sample=False,
                num_beams=beams
            )
        )

    # Disable gradients during inference
    with torch.no_grad():

        return model.generate(**inputs, **kwargs)


# CHOOSE MODEL BASED ON GPU
def pick_model_and_batch(device: str):

    # Read optional environment variables
    env_mid = os.getenv("MODEL_ID", "").strip()
    env_bs = os.getenv("BATCH_SIZE", "").strip()

    # If user already defined model manually
    if env_mid:

        bs = int(env_bs) if env_bs.isdigit() else (
            8 if device == "cuda" else 1
        )

        return env_mid, bs

    # CPU mode
    if device != "cuda":
        return "Qwen/Qwen2.5-VL-3B-Instruct", 1

    # Detect available VRAM
    vram_gb = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024**3)
    )

    # Choose model + batch size automatically
    if vram_gb < 12:
        return "Qwen/Qwen2.5-VL-3B-Instruct", 4

    if vram_gb < 20:
        return "Qwen/Qwen2.5-VL-7B-Instruct", 6

    if vram_gb < 32:
        return "Qwen/Qwen2.5-VL-7B-Instruct", 8

    return "Qwen/Qwen2.5-VL-7B-Instruct", 12


# MAIN FUNCTION
def main():

    # Set random seed
    torch.manual_seed(SEED)

    # Detect device
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # GPU optimization
    if device == "cuda":

        torch.backends.cuda.matmul.allow_tf32 = True

        try:
            torch.set_float32_matmul_precision("high")

        except Exception:
            pass

    # Select model automatically
    model_id, batch_size = pick_model_and_batch(device)

    print(f"[INFO] Device: {device}")
    print(f"[INFO] Model: {model_id}")
    print(f"[INFO] Batch size: {batch_size}")

    # Output file path
    p_out = Path(OUT_JSONL).expanduser().resolve()

    print("[INFO] OUT_JSONL ->", p_out)

    # Store already processed images
    seen = set()

    # Overwrite mode
    if MODE.lower() == "overwrite":

        if p_out.exists():

            print("[INFO] deleting old file")

            p_out.unlink()

    # Find all images
    images_all = find_images(IMAGES_DIR)

    if not images_all:

        raise SystemExit(
            f"No images found in {IMAGES_DIR}"
        )

    print(f"[INFO] Found {len(images_all)} images.")

    # Load processor
    processor = AutoProcessor.from_pretrained(
        model_id,
        trust_remote_code=True,
        use_fast=True
    )

    # Model loading settings
    model_kwargs = dict(trust_remote_code=True)

    # GPU loading
    if device == "cuda":

        model_kwargs.update(
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )

    # CPU loading
    else:

        model_kwargs.update(
            torch_dtype=torch.float32
        )

    # Load model
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        **model_kwargs
    )

    # Evaluation mode
    model.eval()

    # Create output directory
    p_out.parent.mkdir(parents=True, exist_ok=True)

    total = len(images_all)
    wrote = 0

    # Open JSONL output file
    with open(p_out, "a", encoding="utf-8") as fout:

        # Batch processing loop
        for i in range(0, total, batch_size):

            batch_paths = images_all[i:i+batch_size]

            pil_batch = []
            keep_pairs = []

            # Load images
            for p_abs in batch_paths:

                img = load_image_rgb(p_abs)

                if img is None:
                    continue

                pil_batch.append(img)

                keep_pairs.append(p_abs)

            if not keep_pairs:
                continue

            # Prepare prompts
            texts, imgs = prepare_batch_messages_rs(
                pil_batch,
                processor
            )

            # Convert to tensors
            inputs = processor(
                text=texts,
                images=imgs,
                return_tensors="pt",
                padding=True
            )

            # Move tensors to GPU
            for k, v in list(inputs.items()):

                if hasattr(v, "to"):
                    inputs[k] = v.to(device)

            # Generate captions
            out_ids = generate_once(
                model,
                inputs,
                max_new_tokens=MAX_NEW_TOKS,
                sample=False,
                beams=3
            )

            # Remove prompt tokens
            prompt_len = inputs["input_ids"].shape[1]

            # Decode outputs
            decoded = processor.tokenizer.batch_decode(
                out_ids[:, prompt_len:],
                skip_special_tokens=True
            )

            # Clean captions
            caps = [rs_clean_multi(t) for t in decoded]

            # Save JSONL lines
            for p_abs, cap in zip(keep_pairs, caps):

                line = {
                    "image": p_abs.name,
                    "prefix": PREFIX_STR,
                    "suffix": cap
                }

                fout.write(
                    json.dumps(line, ensure_ascii=False)
                    + "\n"
                )

                wrote += 1

            print(
                f"[{min(i+batch_size, total)}/{total}] "
                f"wrote {len(keep_pairs)}"
            )

    print(f"[INFO] Finished. Wrote {wrote} lines")




In [ ]:
# RUN SCRIPT

if __name__ == "__main__":

    main()

[INFO] Device: cuda
[INFO] GPU: NVIDIA A100-SXM4-40GB (39.6 GB)
[INFO] Model: Qwen/Qwen2.5-VL-7B-Instruct
[INFO] Batch size: 12
[INFO] OUT_JSONL -> /content/drive/MyDrive/ALL IN_pg/New Work for Journal /buildings_cap.jsonl
[INFO] Fresh run (no resume).
[INFO] Found 2069 images.


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[DEBUG] raw decode sample: Buildings are present, with a few (1–5) visible. The structures appear to be low-rise, possibly detached houses with gabled roofs. The spatial pattern is linear, following the road alignment. Notable 
[12/2069] wrote 12 (skipped 0)
[24/2069] wrote 12 (skipped 0)
[36/2069] wrote 12 (skipped 0)
[48/2069] wrote 12 (skipped 0)
[60/2069] wrote 12 (skipped 0)
[72/2069] wrote 12 (skipped 0)
[84/2069] wrote 12 (skipped 0)
[96/2069] wrote 12 (skipped 0)
[108/2069] wrote 12 (skipped 0)
[120/2069] wrote 12 (skipped 0)
[132/2069] wrote 12 (skipped 0)
[144/2069] wrote 12 (skipped 0)
[156/2069] wrote 12 (skipped 0)
[168/2069] wrote 12 (skipped 0)
[180/2069] wrote 12 (skipped 0)
[192/2069] wrote 12 (skipped 0)
[204/2069] wrote 12 (skipped 0)
[216/2069] wrote 12 (skipped 0)
[228/2069] wrote 12 (skipped 0)
[240/2069] wrote 12 (skipped 0)
[252/2069] wrote 12 (skipped 0)
[264/2069] wrote 12 (skipped 0)
[276/2069] wrote 12 (skipped 0)
[288/2069] wrote 12 (skipped 0)
[300/2069] w